In [ ]:
import torch

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -U "transformers>=4.43.2" "peft>=0.10.0" "trl>=0.9.0" "accelerate>=0.30.0" bitsandbytes python-dotenv

  Using cached trl-1.0.0-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.8 MB/s eta 0:00:00
Using cached trl-1.0.0-py3-none-any.whl (630 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 136.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing instal

In [ ]:
import json
from datasets import Dataset

# 1. Load your uploaded 3,000 sample dataset
with open("/content/sg_dataset.json") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.15)

# Updated Phase 2: Create a unified 'text' field
def format_example(example):
    instruction = (
        "Please generate a step-by-step solution for the following problem "
        "with no calculations. You don't need to solve it, just output the steps in 2 to 6 steps."
    )
    # Combine everything into one 'text' field
    # Manually add the EOS token here to prevent the list attribute error from before
    full_text = f"Instruction: {instruction}\nQuestion: {example['input']}\nSolution Guidance:\n{example['output']}{tokenizer.eos_token}"
    return {"text": full_text}

dataset = dataset.map(format_example, remove_columns=dataset["train"].column_names)
print("✅ Dataset cleaned: Removed all columns except 'text'")

Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

✅ Dataset cleaned: Removed all columns except 'text'


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from google.colab import userdata

# 1. Environment Configuration
os.environ["MODEL_NAME"] = "Qwen/Qwen2.5-Math-1.5B"
os.environ["OUTPUT_DIR"] = "outputs/qwen_sgft_phase_a"
os.environ["BATCH_SIZE"] = "1"
os.environ["EPOCHS"] = "3"
os.environ["LR"] = "2e-5"

# 2. Load Tokenizer & Model Identifier
model_id = os.environ.get("MODEL_NAME")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 3. 4-bit Quantization (Essential for T4 VRAM)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 4. Model Loading with HF Token [cite: 187, 205]
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    token=hf_token
)

# 5. LoRA Setup for Mathematical Reasoning [cite: 187, 669]
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Phase 3: Updated SFTConfig and Trainer
def formatting_prompts_func(example):
    return example['prompt'] # Return the renamed field

from trl import SFTConfig, SFTTrainer
import os

# 7. Final SFTConfig (Precision Fix)
sft_config = SFTConfig(
    output_dir=os.environ.get("OUTPUT_DIR"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=int(os.environ.get("EPOCHS", 3)),
    learning_rate=float(os.environ.get("LR", 2e-5)),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    # CHANGE THESE TWO LINES:
    fp16=False,        # Disable standard FP16
    bf16=True,         # Enable BF16 for Qwen2.5 stability
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# 2. Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,           # Current library standard
    args=sft_config
)

# 3. RUN TRAINING 🚀
trainer.train()

# 10. Save local adapter
trainer.save_model(os.environ.get("OUTPUT_DIR"))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2550 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2550 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.453961
20,1.357152
30,1.250923
40,1.124533
50,1.044863
60,0.944065
70,0.856265
80,0.714185
90,0.579290
100,0.527760


Step,Training Loss
10,1.453961
20,1.357152
30,1.250923
40,1.124533
50,1.044863
60,0.944065
70,0.856265
80,0.714185
90,0.579290
100,0.527760


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a project folder in your Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/BTech_Project/Phase_A_Weights

# Copy the trained weights from the local Colab disk to your Drive
!cp -r /content/outputs/qwen_sgft_phase_a/* /content/drive/MyDrive/BTech_Project/Phase_A_Weights/

print("✅ Training weights successfully backed up to Google Drive!")

Mounted at /content/drive
✅ Training weights successfully backed up to Google Drive!


In [ ]:
import torch
import gc
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Setup paths (Point to where you saved the weights in Drive)
model_id = "Qwen/Qwen2.5-Math-1.5B"
adapter_path = "/content/drive/MyDrive/BTech_Project/Phase_A_Weights/"

# 2. Load Base Model and Adapter
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# 3. Define the test function
def test_sgft(question):
    instruction = (
        "Please generate a step-by-step solution for the following problem "
        "with no calculations. You don't need to solve it, just output the steps in 2 to 6 steps."
    )
    prompt = f"Instruction: {instruction}\nQuestion: {question}\nSolution Guidance:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1, # Keep it deterministic for logic testing
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("-" * 30)
    print(result.split("Solution Guidance:\n")[-1])
    print("-" * 30)

# 4. Run the test
test_question = "If Kate's 6 friends each eat 6 cheese pizza slices and 4 pepperoni pizza slices, and cheese pizzas have 12 slices while pepperoni have 8, how many total pizzas does she need?"
test_sgft(test_question)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Calculate the total number of cheese pizza slices consumed by multiplying the number of friends by the slices per friend.
Step 2: Determine the total number of pepperoni pizza slices consumed by multiplying the number of friends by the slices per friend.
Step 3: Divide the total cheese pizza slices by the slices per cheese pizza to find the number of cheese pizzas needed.
Step 4: Divide the total pepperoni pizza slices by the slices per pepperoni pizza to find the number of pepperoni pizzas needed.
Step 5: Add the number of cheese and pepperoni pizzas together to find the total number of pizzas required.
------------------------------


In [ ]:
# 1. Define a more diverse set of test cases
stress_tests = [
    {
        "type": "Algebraic/Variables",
        "question": "If x + 5 = 12 and y = 2x, find the value of x + y."
    },
    {
        "type": "Geometry/Area",
        "question": "A rectangular garden has a length of 20m and a width that is half its length. If a 1m path is built inside the perimeter, what is the remaining area of the garden?"
    },
    {
        "type": "Multi-step Logic",
        "question": "A train leaves Station A at 60km/h. Two hours later, a faster train leaves Station A at 90km/h. How many hours will it take for the second train to catch up?"
    },
    {
        "type": "Mixed",
        "question": "A farmer has 15 sheep. All but 8 die. How many sheep are left?"
    }
]

# 2. Execute the Suite
print(f"Testing Model: {model_id} with SGFT Adapters\n")

for test in stress_tests:
    print(f"TEST TYPE: {test['type']}")
    print(f"QUESTION: {test['question']}")
    print("OUTPUT GUIDANCE:")
    test_sgft(test['question']) # Calls the function from the previous step
    print("\n" + "="*50 + "\n")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Testing Model: Qwen/Qwen2.5-Math-1.5B with SGFT Adapters

TEST TYPE: Algebraic/Variables
QUESTION: If x + 5 = 12 and y = 2x, find the value of x + y.
OUTPUT GUIDANCE:


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Solve the first equation for x.
Step 2: Substitute the value of x from Step 1 into the second equation to find y.
Step 3: Add the values of x and y to get the final answer.
------------------------------


TEST TYPE: Geometry/Area
QUESTION: A rectangular garden has a length of 20m and a width that is half its length. If a 1m path is built inside the perimeter, what is the remaining area of the garden?
OUTPUT GUIDANCE:


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Determine the width of the garden by dividing the length by 2.
Step 2: Calculate the total area of the garden by multiplying the length by the width.
Step 3: Calculate the area of the path by multiplying the perimeter of the garden by the width of the path.
Step 4: Subtract the area of the path from the total area of the garden to find the remaining area.
------------------------------


TEST TYPE: Multi-step Logic
QUESTION: A train leaves Station A at 60km/h. Two hours later, a faster train leaves Station A at 90km/h. How many hours will it take for the second train to catch up?
OUTPUT GUIDANCE:


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Determine the distance the first train has traveled by the time the second train starts.
Step 2: Calculate the relative speed between the two trains.
Step 3: Find the time it takes for the second train to cover the distance between them.
------------------------------


TEST TYPE: Mixed
QUESTION: A farmer has 15 sheep. All but 8 die. How many sheep are left?
OUTPUT GUIDANCE:
------------------------------
Step 1: Identify the total number of sheep initially.
Step 2: Determine the number of sheep that did not die.
Step 3: Subtract the number of surviving sheep from the initial total.
------------------------------




In [ ]:
# Phase A: Semantic Planning Stress Test Suite
test_cases = [
    {
        "type": "Linguistic Trap (The Reversal)",
        "question": "A room has 5 people. Each person shakes hands with every other person exactly once. However, two people refuse to shake hands with each other. How many handshakes occur?"
    },
    {
        "type": "Distractor/Irrelevant Info",
        "question": "James has 4 apples and is walking at 5 km/h to a store 2 km away. He meets a friend who gives him 3 more apples. The store sells apples for $1 each. How many apples does James have now?"
    },
    {
        "type": "Unsolvable/Missing Constraint",
        "question": "If a car travels from City A to City B at 60 km/h, what is the total fuel consumed for the trip?"
    },
    {
        "type": "The 'Sheep' Variant (Phrasing)",
        "question": "There are 20 birds on a fence. A hunter shoots one. How many are left on the fence?"
    }
]

def run_advanced_tests(model, tokenizer):
    print(f"--- 🔬 Phase A Stress Test Results ---")
    for test in test_cases:
        print(f"\n[TEST]: {test['type']}")
        print(f"[Q]: {test['question']}")

        # Calling your existing test_sgft function
        test_sgft(test['question'])
        print("-" * 50)

run_advanced_tests(model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


--- 🔬 Phase A Stress Test Results ---

[TEST]: Linguistic Trap (The Reversal)
[Q]: A room has 5 people. Each person shakes hands with every other person exactly once. However, two people refuse to shake hands with each other. How many handshakes occur?


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Determine the total number of handshakes if everyone could shake hands with everyone else.
Step 2: Subtract the handshake that the two individuals who refuse to shake hands are supposed to have.
------------------------------
--------------------------------------------------

[TEST]: Distractor/Irrelevant Info
[Q]: James has 4 apples and is walking at 5 km/h to a store 2 km away. He meets a friend who gives him 3 more apples. The store sells apples for $1 each. How many apples does James have now?


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Determine the total number of apples James has after receiving more from his friend.
Step 2: Calculate the total number of apples James has after the store sells them.
------------------------------
--------------------------------------------------

[TEST]: Unsolvable/Missing Constraint
[Q]: If a car travels from City A to City B at 60 km/h, what is the total fuel consumed for the trip?


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


------------------------------
Step 1: Determine the distance between City A and City B.
Step 2: Calculate the time it takes to travel this distance at the given speed.
Step 3: Find the fuel consumption rate per hour.
Step 4: Multiply the time by the fuel consumption rate to get the total fuel used.
------------------------------
--------------------------------------------------

[TEST]: The 'Sheep' Variant (Phrasing)
[Q]: There are 20 birds on a fence. A hunter shoots one. How many are left on the fence?
------------------------------
Step 1: Identify the initial number of birds on the fence.
Step 2: Determine the action that occurs (the hunter shooting one bird).
Step 3: Subtract the number of birds shot from the initial count to find the remaining number.
------------------------------
--------------------------------------------------
